# Urban Heat & Cooling-Priority Mapping — Track B / RF1: Random Forest Baseline

**NUS-ISS Practice Module, Week 2-3, Step 3 (RF first — fast, establishes the floor).**

Trains a Random Forest land-cover classifier on ESA WorldCover labels (collapsed
to the 4-class gate-review scheme: vegetation / built_up / bare / water), using
Sentinel-2 bands + NDVI/NDBI/NDWI as features, then classifies the full
Singapore composite into a land-cover raster.

**Non-circularity, enforced not assumed:** training pixels are drawn from
Singapore's land area *excluding a buffer around every one of your 200
validation points* (RF1.5) — so it's not just "we didn't intentionally
include them," there's an explicit spatial exclusion check with a printed
count of how much area got excluded.

**What this notebook does NOT do:** the formal accuracy evaluation (confusion
matrix, per-class F1, comparison table) — that's a separate notebook run once
U-Net and the ensemble also exist, so all three get evaluated identically.
RF1.9 here is only a quick informal sanity-check accuracy, not the deliverable.

Run top to bottom. Uses the same locked config as `generate_validation_sample.ipynb`
(SB1) — season window, CRS, WorldCover version, boundary fetch — kept in sync
deliberately; if you change one, change the other.

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [1]:
# --- SETUP CELL 1: Install dependencies -------------------------------------
!pip install -q earthengine-api geemap pandas geopandas requests scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 29.4 MB/s eta 0:00:00


## Setup 2 — Authenticate & initialize Earth Engine

In [2]:
# --- SETUP CELL 2: Authenticate & initialize Earth Engine -------------------
import ee

PROJECT_ID = "nus-iss-urban-heat-sg"  # <-- must match SB1 / Track A's project

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("EE initialized OK, project:", PROJECT_ID)


EE initialized OK, project: nus-iss-urban-heat-sg


## Setup 3 — Mount Google Drive

In [3]:
# --- SETUP CELL 3: Mount Google Drive ----------------------------------------
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


Mounted at /content/drive
Drive mounted at /content/drive


## Setup 4 — Initialize shared results tracker

In [4]:
# --- SETUP CELL 4: Initialize shared results tracker ------------------------
track_b_results = {}
print("track_b_results initialized")


track_b_results initialized


---
# RF1 — Random Forest baseline (train + classify)


## RF1.1 — Config

In [5]:
# --- RF1 CELL 1: Config -------------------------------------------------------
sg_bbox = ee.Geometry.Rectangle([103.55, 1.15, 104.10, 1.48])

# Locked C4 season window — IDENTICAL to SB1 / gee_heat_variants.ipynb /
# adaptive_capacity_pillar.ipynb. Do not drift this independently.
YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
DRY_SEASON_MONTHS = [4, 5, 10, 11]
S2_CLOUD_PROB_MAX = 70   # matches Track A's locked C4 CLOUD_COVER_MAX (confirmed via their real run)

TARGET_SCALE = 10
S2_UTM_CRS = "EPSG:32648"

WORLDCOVER_ASSET = "ESA/WorldCover/v200/2021"
SUBZONE_DATASET_ID = "d_8594ae9ff96d0c708bc2af633048edfb"

# WorldCover v200 native class codes -> 4-class bucket scheme (IDENTICAL to
# SB1.1's WC_TO_BUCKET_FROM/TO — keep in sync if either changes)
WC_TREE, WC_SHRUB, WC_GRASS, WC_CROP = 10, 20, 30, 40
WC_BUILTUP, WC_BARE, WC_SNOWICE, WC_WATER = 50, 60, 70, 80
WC_WETLAND, WC_MANGROVE, WC_MOSSLICHEN = 90, 95, 100

BUCKET_VEGETATION, BUCKET_BUILTUP, BUCKET_BARE, BUCKET_WATER = 1, 2, 3, 4
BUCKET_NAMES = {
    BUCKET_VEGETATION: "vegetation", BUCKET_BUILTUP: "built_up",
    BUCKET_BARE: "bare", BUCKET_WATER: "water",
}
WC_TO_BUCKET_FROM = [WC_TREE, WC_SHRUB, WC_GRASS, WC_CROP, WC_BUILTUP,
                      WC_BARE, WC_SNOWICE, WC_WATER, WC_WETLAND,
                      WC_MANGROVE, WC_MOSSLICHEN]
WC_TO_BUCKET_TO = [BUCKET_VEGETATION, BUCKET_VEGETATION, BUCKET_VEGETATION,
                    BUCKET_VEGETATION, BUCKET_BUILTUP, BUCKET_BARE, 0,
                    BUCKET_WATER, BUCKET_VEGETATION, BUCKET_VEGETATION,
                    BUCKET_VEGETATION]

# Feature bands: raw Sentinel-2 + NDVI/NDBI/NDWI, per the locked project spec
S2_FEATURE_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]  # Blue, Green, Red, NIR, SWIR1, SWIR2
INDEX_BANDS = ["NDVI", "NDBI", "NDWI"]
ALL_FEATURE_BANDS = S2_FEATURE_BANDS + INDEX_BANDS

# Validation point exclusion buffer (non-circularity enforcement)
VALIDATION_EXCLUSION_BUFFER_M = 15   # ~1.5x pixel — errs generous, not tight

# Training sampling
TRAINING_POINTS_PER_CLASS = 3000   # cap; stratifiedSample returns fewer if unavailable
RANDOM_SEED = 42

# RF hyperparameters (ee.Classifier.smileRandomForest)
RF_NUM_TREES = 200
RF_MIN_LEAF_POPULATION = 1
RF_BAG_FRACTION = 0.5

DRIVE_DIR = "/content/drive/MyDrive/urban_heat_sg"
VALIDATION_CSV = f"{DRIVE_DIR}/validation_sample_200_labeled.csv"  # your current labeled set
RF_RASTER_EXPORT_DESCRIPTION = "rf_landcover_classified"
RF_RASTER_EXPORT_FOLDER = "urban_heat_sg"

print("Feature bands:", ALL_FEATURE_BANDS)
print("Validation CSV:", VALIDATION_CSV)
print(f"Training cap: {TRAINING_POINTS_PER_CLASS}/class, RF trees: {RF_NUM_TREES}, seed: {RANDOM_SEED}")


Feature bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI', 'NDWI']
Validation CSV: /content/drive/MyDrive/urban_heat_sg/validation_sample_200_labeled.csv
Training cap: 3000/class, RF trees: 200, seed: 42


## RF1.2 — Fetch the real Singapore boundary (same as SB1.2)

In [6]:
# --- RF1 CELL 2: Fetch Singapore boundary ------------------------------------
import requests
import json as _json

def fetch_datagovsg_geojson(dataset_id, out_path):
    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    r = requests.get(poll_url)
    r.raise_for_status()
    payload = r.json()
    if payload.get("code") != 0:
        raise RuntimeError(f"data.gov.sg API error: {payload.get('errMsg')}")
    geojson_bytes = requests.get(payload["data"]["url"]).content
    with open(out_path, "wb") as f:
        f.write(geojson_bytes)
    return out_path

subzone_path = fetch_datagovsg_geojson(SUBZONE_DATASET_ID, "/content/ura_subzones_rf1.geojson")
with open(subzone_path) as f:
    subzone_geojson = _json.load(f)

for feature in subzone_geojson.get("features", []):
    props = feature.get("properties", {})
    for old_key in list(props.keys()):
        if "." in old_key:
            props[old_key.replace(".", "_")] = props.pop(old_key)

subzones = ee.FeatureCollection(subzone_geojson)
sg_boundary = subzones.union(1).first().geometry()
print(f"Singapore boundary built from {subzones.size().getInfo()} subzones.")


Singapore boundary built from 332 subzones.


## RF1.3 — Cloud masking + season-filter helpers (same as SB1.3)

In [7]:
# --- RF1 CELL 3: Cloud masking + season-filter helpers ----------------------
def mask_s2_clouds(cloud_prob_image):
    return cloud_prob_image.select("probability").lt(S2_CLOUD_PROB_MAX)


def date_filter_for_years_months(collection, years, months):
    filters = []
    for y in years:
        for m in months:
            start = ee.Date.fromYMD(y, m, 1)
            end = start.advance(1, "month")
            filters.append(ee.Filter.date(start, end))
    return collection.filter(ee.Filter.Or(*filters))


print("Helpers defined.")


Helpers defined.


## RF1.4 — Build Sentinel-2 composite + spectral indices (season-controlled, C4)

Same season-controlled composite approach as SB1, but this time keeping the
actual band values (not just the mask) since these are the classifier's
input features. Indices computed from the composite (index-of-median), not
per-scene then medianed — simpler, and a defensible standard choice; note
this in your methods if it matters for reproducing exactly.


In [8]:
# --- RF1 CELL 4: Sentinel-2 composite + indices -------------------------------
s2_sr = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(sg_bbox)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70))
)
s2_sr = date_filter_for_years_months(s2_sr, YEARS, DRY_SEASON_MONTHS)
print("Sentinel-2 scenes after season filter (pre-mask):", s2_sr.size().getInfo())

s2_cloud_prob = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(sg_bbox)
s2_cloud_prob = date_filter_for_years_months(s2_cloud_prob, YEARS, DRY_SEASON_MONTHS)

joined = ee.Join.saveFirst("cloud_mask").apply(
    primary=s2_sr, secondary=s2_cloud_prob,
    condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
)

def _mask(img):
    img = ee.Image(img)
    cloud_img = ee.Image(img.get("cloud_mask"))
    return img.updateMask(mask_s2_clouds(cloud_img))

s2_masked = ee.ImageCollection(joined).map(_mask)
print("Sentinel-2 usable scenes (post-mask):", s2_masked.size().getInfo())

composite_bands = s2_masked.select(S2_FEATURE_BANDS).median().clip(sg_boundary)
composite_bands = composite_bands.reproject(crs=S2_UTM_CRS, scale=TARGET_SCALE)

ndvi = composite_bands.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndbi = composite_bands.normalizedDifference(["B11", "B8"]).rename("NDBI")
ndwi = composite_bands.normalizedDifference(["B3", "B8"]).rename("NDWI")

feature_image = ee.Image.cat([composite_bands, ndvi, ndbi, ndwi]).select(ALL_FEATURE_BANDS)
valid_mask = feature_image.select("B4").mask()

proj_check = feature_image.projection().getInfo()
print(f"Feature image projection: {proj_check['crs']} (expect {S2_UTM_CRS})")
print(f"Bands: {feature_image.bandNames().getInfo()}")


Sentinel-2 scenes after season filter (pre-mask): 69
Sentinel-2 usable scenes (post-mask): 69
Feature image projection: EPSG:32648 (expect EPSG:32648)
Bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI', 'NDWI']


## RF1.5 — WorldCover training labels (4-class) + validation-point exclusion

Collapses WorldCover the same way SB1 does, then builds the actual training
region: Singapore's boundary MINUS a buffer around every one of your 200
validation points. This is what makes the no-circularity claim checkable,
not just asserted.


In [9]:
# --- RF1 CELL 5: Training labels + validation exclusion -----------------------
import pandas as pd

worldcover_raw = ee.Image(WORLDCOVER_ASSET).select("Map")
worldcover = worldcover_raw.reproject(crs=S2_UTM_CRS, scale=TARGET_SCALE).clip(sg_boundary)

wc_bucket = worldcover.remap(WC_TO_BUCKET_FROM, WC_TO_BUCKET_TO, 0).rename("wc_class")
wc_bucket = wc_bucket.updateMask(wc_bucket.neq(0)).updateMask(valid_mask)

# Load validation points and build the exclusion buffer.
val_df = pd.read_csv(VALIDATION_CSV)
print(f"Loaded {len(val_df)} validation points from {VALIDATION_CSV}")

val_geoms = [ee.Geometry.Point([row.lon, row.lat]) for row in val_df.itertuples()]
val_points_fc = ee.FeatureCollection([ee.Feature(g) for g in val_geoms])
val_buffer = val_points_fc.geometry().buffer(VALIDATION_EXCLUSION_BUFFER_M)

training_region = sg_boundary.difference(val_buffer, ee.ErrorMargin(1))

# Sanity check: confirm the exclusion actually removed area (not a no-op).
sg_area = sg_boundary.area(1).getInfo()
training_area = training_region.area(1).getInfo()
excluded_area = sg_area - training_area
print(f"Singapore boundary area: {sg_area/1e6:,.2f} km²")
print(f"Training region area (post-exclusion): {training_area/1e6:,.2f} km²")
print(f"Excluded around validation points: {excluded_area:,.0f} m² "
      f"(~{excluded_area / (3.14159 * VALIDATION_EXCLUSION_BUFFER_M**2):.0f} point-buffers' worth)")
if excluded_area < 1000:
    print("⚠️  Exclusion area suspiciously small — check val_df loaded correctly before trusting this.")
else:
    print("✅ Validation points are spatially excluded from the training region.")


Loaded 200 validation points from /content/drive/MyDrive/urban_heat_sg/validation_sample_200_labeled.csv
Singapore boundary area: 788.30 km²
Training region area (post-exclusion): 788.16 km²
Excluded around validation points: 139,654 m² (~198 point-buffers' worth)
✅ Validation points are spatially excluded from the training region.


## RF1.6 — Extract training samples (stratified by class, capped per class)

Note: we do NOT pull all training records locally with `.getInfo()` — Earth
Engine caps synchronous FeatureCollection pulls at 5000 elements, and with
4 classes x up to 3000 each that's already at 12000. `aggregate_histogram()`
computes the per-class counts server-side instead, returning a small dict
rather than every feature's band values.


In [10]:
# --- RF1 CELL 6: Extract training samples --------------------------------------
training_image = ee.Image.cat([feature_image, wc_bucket])

class_values = list(BUCKET_NAMES.keys())
class_points = [TRAINING_POINTS_PER_CLASS] * len(class_values)

training_fc = training_image.stratifiedSample(
    numPoints=0,
    classBand="wc_class",
    region=training_region,
    scale=TARGET_SCALE,
    classValues=class_values,
    classPoints=class_points,
    seed=RANDOM_SEED,
    geometries=False,   # don't need geometry back, just band values — cheaper pull
    dropNulls=True,
    tileScale=8,
)

n_training = training_fc.size().getInfo()
print(f"Training samples drawn: {n_training} (requested up to {sum(class_points)})")

# Server-side count per class — avoids the 5000-element getInfo() cap that
# pulling all individual feature records would hit.
train_counts_raw = training_fc.aggregate_histogram("wc_class").getInfo()
train_counts = {int(k): v for k, v in train_counts_raw.items()}

print("Training samples per class:")
for cls, n in sorted(train_counts.items(), key=lambda kv: -kv[1]):
    name = BUCKET_NAMES.get(cls, f"class_{cls}")
    print(f"  {cls} {name:<12} {n}")


Training samples drawn: 12000 (requested up to 12000)
Training samples per class:
  1 vegetation   3000
  2 built_up     3000
  3 bare         3000
  4 water        3000


## RF1.7 — Train Random Forest (ee.Classifier.smileRandomForest)

In [11]:
# --- RF1 CELL 7: Train RF ------------------------------------------------------
rf_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=RF_NUM_TREES,
    minLeafPopulation=RF_MIN_LEAF_POPULATION,
    bagFraction=RF_BAG_FRACTION,
    seed=RANDOM_SEED,
).train(
    features=training_fc,
    classProperty="wc_class",
    inputProperties=ALL_FEATURE_BANDS,
)

# Quick check the classifier trained without error and has expected schema.
schema = rf_classifier.schema().getInfo()
print(f"RF trained. Input properties: {schema}")
print(f"Trees: {RF_NUM_TREES}, min leaf population: {RF_MIN_LEAF_POPULATION}, "
      f"bag fraction: {RF_BAG_FRACTION}, seed: {RANDOM_SEED}")


RF trained. Input properties: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI', 'NDWI']
Trees: 200, min leaf population: 1, bag fraction: 0.5, seed: 42


## RF1.8 — Classify the full composite

In [12]:
# --- RF1 CELL 8: Classify -------------------------------------------------------
rf_classified = feature_image.classify(rf_classifier).rename("rf_class").clip(sg_boundary)
print("Classification complete (lazy — actual computation happens on export/sample).")
print(f"Output band: {rf_classified.bandNames().getInfo()}")


Classification complete (lazy — actual computation happens on export/sample).
Output band: ['rf_class']


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


## RF1.9 — Quick informal accuracy check (NOT the formal Step-4 evaluation)

Samples the classified raster at your 200 validation points and compares to
`agreed_label`, purely as a sanity check that the RF baseline is in a
reasonable ballpark before moving on. The real confusion matrix / per-class
F1 / comparison table happens in a separate evaluation notebook once U-Net
and the ensemble also exist, so all three are scored identically.


In [13]:
# --- RF1 CELL 9: Informal accuracy check ----------------------------------------
val_df["agreed_label"] = val_df["agreed_label"].fillna("").astype(str)
valid_rows = val_df[val_df["agreed_label"].isin(BUCKET_NAMES.values())].copy()
n_skipped = len(val_df) - len(valid_rows)
if n_skipped > 0:
    print(f"Skipping {n_skipped} validation point(s) with blank/uncertain labels for this check.")

val_fc_labeled = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([row.lon, row.lat]), {"point_id": row.point_id})
    for row in valid_rows.itertuples()
])

sampled = rf_classified.sampleRegions(
    collection=val_fc_labeled, scale=TARGET_SCALE, geometries=False, tileScale=4,
)
sampled_records = sampled.getInfo()["features"]

pred_by_id = {f["properties"]["point_id"]: f["properties"].get("rf_class") for f in sampled_records}
BUCKET_NAME_TO_ID = {v: k for k, v in BUCKET_NAMES.items()}

valid_rows["rf_pred_bucket"] = valid_rows["point_id"].map(pred_by_id)
valid_rows["rf_pred_name"] = valid_rows["rf_pred_bucket"].map(BUCKET_NAMES)
valid_rows["true_bucket"] = valid_rows["agreed_label"].map(BUCKET_NAME_TO_ID)

n_missing_pred = valid_rows["rf_pred_bucket"].isna().sum()
if n_missing_pred > 0:
    print(f"⚠️  {n_missing_pred} point(s) got no prediction (likely masked/no-data pixel).")

scored = valid_rows.dropna(subset=["rf_pred_bucket"])
accuracy = (scored["rf_pred_bucket"] == scored["true_bucket"]).mean()
print(f"\nInformal RF accuracy on {len(scored)} validation points: {accuracy*100:.1f}%")
print("(Sanity check only — run the formal evaluation notebook for the real deliverable numbers.)")

print("\nQuick cross-tab (rows=true, columns=predicted):")
print(pd.crosstab(scored["agreed_label"], scored["rf_pred_name"]))



Informal RF accuracy on 200 validation points: 72.5%
(Sanity check only — run the formal evaluation notebook for the real deliverable numbers.)

Quick cross-tab (rows=true, columns=predicted):
rf_pred_name  bare  built_up  vegetation  water
agreed_label                                   
bare             8         0           1      4
built_up        25        58           6      1
vegetation       4        12          61      1
water            0         1           0     18


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


## RF1.10 — Export classified raster to Drive

Standard EE async export — this cell submits the task and polls until done
(can take a few minutes for a full-Singapore raster at 10m).


In [14]:
# --- RF1 CELL 10: Export raster -------------------------------------------------
import time

export_task = ee.batch.Export.image.toDrive(
    image=rf_classified,
    description=RF_RASTER_EXPORT_DESCRIPTION,
    folder=RF_RASTER_EXPORT_FOLDER,
    region=sg_boundary,
    scale=TARGET_SCALE,
    crs=S2_UTM_CRS,
    maxPixels=1e13,
    fileFormat="GeoTIFF",
)
export_task.start()
print(f"Export task started: {RF_RASTER_EXPORT_DESCRIPTION}")

while export_task.active():
    status = export_task.status()
    print(f"  ...{status['state']}")
    time.sleep(15)

final_status = export_task.status()
print(f"\nFinal status: {final_status['state']}")
if final_status["state"] == "COMPLETED":
    print(f"✅ Exported to Drive: {RF_RASTER_EXPORT_FOLDER}/{RF_RASTER_EXPORT_DESCRIPTION}.tif")
else:
    print(f"⚠️  Export did not complete cleanly: {final_status}")


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Export task started: rf_landcover_classified
  ...READY
  ...READY
  ...READY
  ...READY
  ...READY
  ...READY
  ...READY
  ...READY
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING
  ...RUNNING

Final status: COMPLETED
✅ Exported to Drive: urban_heat_sg/rf_landcover_classified.tif


## RF1.11 — Verdict

In [15]:
# --- RF1 CELL 11: Verdict --------------------------------------------------------
print("\n--- RF1 Verdict ---")

rf1_checks = {
    "Feature image built with expected bands": len(feature_image.bandNames().getInfo()) == len(ALL_FEATURE_BANDS),
    "Validation points spatially excluded from training": excluded_area >= 1000,
    "Training samples drawn for all 4 classes": len(train_counts) == 4,
    "RF trained successfully": schema is not None,
    "Raster export completed": final_status["state"] == "COMPLETED",
    "Informal accuracy computed": len(scored) > 0,
}

for check, passed in rf1_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

rf1_pass = all(rf1_checks.values())
print(f"\n{'✅ RF1 PASS' if rf1_pass else '⚠️  RF1 FAIL — review flagged checks'}")

track_b_results["RF1_baseline"] = {
    "status": "PASS" if rf1_pass else "FAIL",
    "checks": rf1_checks,
    "n_training_samples": n_training,
    "training_class_counts": train_counts,
    "informal_accuracy": float(accuracy) if 'accuracy' in dir() else None,
    "rf_hyperparams": {
        "num_trees": RF_NUM_TREES, "min_leaf_population": RF_MIN_LEAF_POPULATION,
        "bag_fraction": RF_BAG_FRACTION, "seed": RANDOM_SEED,
    },
}



--- RF1 Verdict ---
  [PASS] Feature image built with expected bands
  [PASS] Validation points spatially excluded from training
  [PASS] Training samples drawn for all 4 classes
  [PASS] RF trained successfully
  [PASS] Raster export completed
  [PASS] Informal accuracy computed

✅ RF1 PASS
